# Generate DPO Pairs — SFT Model + Claude Judge
**Model:** `SamQuiring/qwen3-8b-tv-tower` (your SFT checkpoint)  
**Output:** `berlin_tv_tower_dpo.jsonl` — one row per prompt with `prompt`, `chosen`, `rejected`  
**Next step:** `qwen3_dpo_finetuning.ipynb`

### Why this is the right approach
In real DPO, both `chosen` and `rejected` must come from the **policy model's own distribution** — not from a different model. We:
1. Generate two candidates per prompt from the SFT model at high temperature (natural variance)
2. Use Claude as a judge to rank them on persona fidelity, tone, and factual accuracy
3. The winner becomes `chosen`, the loser `rejected`

This gives DPO a real job: discriminating between two plausible-but-unequal tower responses, not just re-learning what SFT already taught.

## 1. Environment Check

In [ ]:
import torch

print(f"Torch version:  {torch.__version__}")
print(f"CUDA version:   {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install Anthropic SDK if not present
import importlib
if importlib.util.find_spec("anthropic") is None:
    import subprocess
    subprocess.run(["uv", "pip", "install", "anthropic"], check=True)

import anthropic
print(f"Anthropic SDK version: {anthropic.__version__}")

## 2. Load SFT Model

We generate both candidates from `SamQuiring/qwen3-8b-tv-tower` — the same model we'll train with DPO.  
High temperature (1.0) creates natural variance between two runs of the same prompt.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096
SFT_MODEL = "SamQuiring/qwen3-8b-tv-tower"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
)
FastLanguageModel.for_inference(model)

print(f"Model loaded: {SFT_MODEL}")
print(f"VRAM after load: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

## 3. Load Prompts

We reuse the prompts from the SFT training data — same questions, now generating two candidate answers from the trained model.

In [ ]:
import json

DATASET_PATH = "../berlin_tv_tower_training.jsonl"

prompts = []
with open(DATASET_PATH) as f:
    for line in f:
        row = json.loads(line)
        user_turn = next(m for m in row["messages"] if m["role"] == "user")
        prompts.append(user_turn["content"])

print(f"Loaded {len(prompts):,} prompts")
print(f"\nSample prompts:")
for p in prompts[:5]:
    print(f"  - {p}")

## 4. Generate Two Candidates per Prompt

Temperature 1.0 gives enough variance that one response is often noticeably better than the other — more historically accurate, more in-character, better phrased.  
Both responses still speak as the tower (the SFT already locked that in), so the judge is evaluating *quality within the persona*, not presence of the persona.

~490 generations at 200 tokens each — roughly 8–12 minutes on an H100.

In [ ]:
def generate_response(prompt, max_new_tokens=200, temperature=1.0):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()


print("Generating candidate A and B for each prompt...")
candidates = []
for i, prompt in enumerate(prompts):
    response_a = generate_response(prompt)
    response_b = generate_response(prompt)
    candidates.append({"prompt": prompt, "response_a": response_a, "response_b": response_b})
    if (i + 1) % 25 == 0 or i == 0:
        print(f"  [{i+1}/{len(prompts)}] {prompt[:55]}")
        print(f"    A: {response_a[:80]}...")
        print(f"    B: {response_b[:80]}...")

print(f"\nGenerated {len(candidates) * 2} total responses.")

## 5. Claude as Judge

For each pair, Claude picks which response better embodies the Berlin TV Tower — evaluating:
- **Persona fidelity**: speaks as the tower in first person, never breaks character
- **Factual accuracy**: correct height, dates, history
- **Tone**: confident, grounded, slightly philosophical — not sycophantic or hollow

Claude returns `{"preferred": "A" or "B", "reason": "..."}`.  
We randomize A/B presentation per prompt to avoid position bias.

In [ ]:
import os
import random
import re

# Set your API key: export ANTHROPIC_API_KEY=...
claude = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

JUDGE_SYSTEM = """\
You are evaluating two responses from a chatbot that roleplays as the Berlin TV Tower (Fernsehturm).
The tower speaks in first person, is proud of its history and engineering, and never breaks character.

Key facts: 368m tall, built 1965-1969 by East Germany, located at Alexanderplatz, sphere at 203m
contains observation deck + rotating restaurant, ~1.2 million visitors/year.

Evaluate on:
1. Persona fidelity — speaks as the tower, never as an AI
2. Factual accuracy — correct details about the tower
3. Tone — confident, direct, grounded; not hollow or generic

Respond with valid JSON only: {"preferred": "A" or "B", "reason": "one sentence"}"""


def judge_pair(prompt, response_a, response_b):
    user_msg = f"""Prompt: {prompt}

Response A:
{response_a}

Response B:
{response_b}

Which response is better?"""

    message = claude.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=128,
        system=JUDGE_SYSTEM,
        messages=[{"role": "user", "content": user_msg}],
    )
    raw = message.content[0].text.strip()
    # Strip markdown code fences if present
    raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw, flags=re.MULTILINE).strip()
    return json.loads(raw)


print("Running Claude judge on all pairs...")
rng = random.Random(42)
dpo_pairs = []
judge_errors = []

for i, c in enumerate(candidates):
    # Randomize presentation order to reduce position bias
    flip = rng.random() < 0.5
    a = c["response_b"] if flip else c["response_a"]
    b = c["response_a"] if flip else c["response_b"]

    try:
        result = judge_pair(c["prompt"], a, b)
        preferred_is_a = result["preferred"] == "A"
        chosen  = a if preferred_is_a else b
        rejected = b if preferred_is_a else a
        dpo_pairs.append({
            "prompt":   c["prompt"],
            "chosen":   chosen,
            "rejected": rejected,
            "reason":   result.get("reason", ""),
        })
    except Exception as e:
        judge_errors.append({"index": i, "error": str(e)})

    if (i + 1) % 25 == 0 or i == 0:
        print(f"  [{i+1}/{len(candidates)}] judged — {len(judge_errors)} errors so far")

print(f"\nJudged {len(dpo_pairs)} pairs ({len(judge_errors)} skipped due to errors)")

## 6. Preview Pairs

In [ ]:
for pair in dpo_pairs[:3]:
    print(f"{'='*70}")
    print(f"PROMPT:   {pair['prompt']}")
    print(f"REASON:   {pair['reason']}")
    print(f"{'-'*70}")
    print(f"CHOSEN:   {pair['chosen'][:250]}")
    print(f"{'-'*70}")
    print(f"REJECTED: {pair['rejected'][:250]}")

## 7. Save DPO Dataset

TRL's `DPOTrainer` conversational format:
```json
{
  "prompt":   [{"role": "user",      "content": "..."}],
  "chosen":   [{"role": "assistant", "content": "..."}],
  "rejected": [{"role": "assistant", "content": "..."}]
}
```

In [ ]:
OUTPUT_PATH = "berlin_tv_tower_dpo.jsonl"

with open(OUTPUT_PATH, "w") as f:
    for pair in dpo_pairs:
        row = {
            "prompt":   [{"role": "user",      "content": pair["prompt"]}],
            "chosen":   [{"role": "assistant", "content": pair["chosen"]}],
            "rejected": [{"role": "assistant", "content": pair["rejected"]}],
        }
        f.write(json.dumps(row) + "\n")

print(f"Saved {len(dpo_pairs):,} DPO pairs to {OUTPUT_PATH}")

with open(OUTPUT_PATH) as f:
    sample = json.loads(f.readline())
print(f"Columns: {list(sample.keys())}")

## Done

`berlin_tv_tower_dpo.jsonl` is ready. Open `qwen3_dpo_finetuning.ipynb` to train.